# 第20章　画像品質とアーティファクト ― 「悪い画像」をどう扱うか

**『医療診断支援AI開発　社会実装編 ― 臨床現場に届ける（社会実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-social

## 品質スコアを、指標で組む

In [ ]:
def quality_flags(img, hdr, modality):
    """閾値はモダリティごとに分ける。CTとMRでは画素値の意味も分布もまったく違う。"""
    th = THRESHOLDS[modality]          # {"blur":..., "contrast":...} をモダリティ別に持つ
    flags = []
    # 1) ボケ: ラプラシアン分散が低い=ボケ/のっぺり。ただしノイズが多いと逆に上がるので、
    #    「値が高い＝鮮明」とは限らない。低い側だけを弾く用途に留める。
    if laplacian_var(img) < th["blur"]:
        flags.append("blurry")
    # 2) コントラスト不足: 画素分布の広がりが狭い。解剖や表示窓にも左右されるため、
    #    単一の閾値を「読影可否の保証」として扱わない。
    if img.std() < th["contrast"]:
        flags.append("low_contrast")
    # 3) 撮影条件の不一致: 造影相・スライス厚を確認。
    #    造影相を直接示す標準タグは無い。ContrastBolusAgent(0018,0010) や
    #    遅延時間・シリーズ記述から、施設ごとのマッピングで推定するしかない。
    if estimate_phase(hdr) != EXPECTED_PHASE:
        flags.append("wrong_phase")
    # 4) 対象範囲外れ: 想定臓器の存在をラフな検出器で確認
    if not organ_present(img):
        flags.append("out_of_fov")
    return flags